# INT8/FP16 Quantization: Efficient Model Deployment

This notebook explores **model quantization**, one of the most practical techniques for deploying neural networks efficiently. We'll cover INT8 and FP16 quantization - reducing model weights and activations from 32-bit floats to lower precision while maintaining accuracy.

**What you'll learn:**
- Why quantization matters for deployment (memory, speed, energy)
- Numerical representations: FP32 vs FP16 vs INT8
- Post-Training Quantization (PTQ): quantize without retraining
- Quantization-Aware Training (QAT): train with quantization in mind
- Dynamic vs static quantization tradeoffs
- Implementing quantization from scratch and with PyTorch

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from aiml_notebooks import get_device, set_seed, create_dataset, create_dataloaders, get_dataset_config

%load_ext autoreload
%autoreload 2

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## Part 1: Why Quantization Matters

### The Deployment Challenge

Neural networks are trained using **FP32** (32-bit floating point), but this creates deployment challenges:

| Model | FP32 Size | Memory Bandwidth | Inference Cost |
|-------|-----------|------------------|----------------|
| ResNet-50 | 98 MB | High | $$$ |
| BERT-Base | 440 MB | Very High | $$$$ |
| GPT-2 | 548 MB | Very High | $$$$ |
| LLaMA-7B | 26 GB | Extreme | $$$$$ |

**Quantization** reduces these numbers by using lower precision:
- **FP16**: 2x smaller, 2x faster memory access
- **INT8**: 4x smaller, 4x faster memory access
- **INT4**: 8x smaller, specialized hardware support

Use a tiny numerical example to see when reduced precision stops preserving small updates.

In [ ]:
# Visualize the memory savings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Memory comparison
precisions = ['FP32\n(32-bit)', 'FP16\n(16-bit)', 'INT8\n(8-bit)', 'INT4\n(4-bit)']
bits = [32, 16, 8, 4]
colors = ['#e74c3c', '#f39c12', '#27ae60', '#3498db']

bars = axes[0].bar(precisions, bits, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_ylabel('Bits per Value', fontsize=12)
axes[0].set_title('Memory per Value by Precision', fontsize=14, fontweight='bold')
axes[0].axhline(y=32, color='gray', linestyle='--', alpha=0.3)

for bar, b in zip(bars, bits):
    reduction = 32 / b
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{reduction:.0f}x', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Right: Example model sizes
models = ['ResNet-50', 'BERT-Base', 'GPT-2', 'LLaMA-7B']
fp32_sizes = [98, 440, 548, 26000]  # MB

x = np.arange(len(models))
width = 0.2

axes[1].bar(x - 1.5*width, fp32_sizes, width, label='FP32', color='#e74c3c', alpha=0.8)
axes[1].bar(x - 0.5*width, [s/2 for s in fp32_sizes], width, label='FP16', color='#f39c12', alpha=0.8)
axes[1].bar(x + 0.5*width, [s/4 for s in fp32_sizes], width, label='INT8', color='#27ae60', alpha=0.8)
axes[1].bar(x + 1.5*width, [s/8 for s in fp32_sizes], width, label='INT4', color='#3498db', alpha=0.8)

axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('Size (MB)', fontsize=12)
axes[1].set_title('Model Size at Different Precisions', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, fontsize=10)
axes[1].legend(fontsize=10)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("Key insight: INT8 quantization provides 4x memory reduction!")
print("This means a 7B parameter model (26GB) fits in just 6.5GB.")

## Part 2: Understanding Numerical Representations

### Floating Point Numbers (FP32, FP16)

Floating point uses three components:
- **Sign bit**: 0 = positive, 1 = negative
- **Exponent**: Determines the magnitude (powers of 2)
- **Mantissa** (fraction): Determines precision

| Format | Sign | Exponent | Mantissa | Total | Range | Precision |
|--------|------|----------|----------|-------|-------|----------|
| FP32 | 1 | 8 | 23 | 32 bits | +/-3.4e38 | ~7 digits |
| FP16 | 1 | 5 | 10 | 16 bits | +/-65504 | ~3 digits |
| BF16 | 1 | 8 | 7 | 16 bits | +/-3.4e38 | ~2 digits |

**BFloat16** keeps FP32's exponent range but reduces precision - great for training!

In [ ]:
def analyze_float_precision(dtype, name):
    """Analyze properties of a floating point dtype."""
    info = torch.finfo(dtype)
    return {
        'name': name,
        'bits': info.bits,
        'max': info.max,
        'min': info.tiny,
        'eps': info.eps,
    }

# Compare floating point formats
fp_formats = [
    analyze_float_precision(torch.float32, 'FP32'),
    analyze_float_precision(torch.float16, 'FP16'),
    analyze_float_precision(torch.bfloat16, 'BF16'),
]

print("Floating Point Format Comparison:")
print("=" * 70)
print(f"{'Format':<10} {'Bits':<8} {'Max Value':<15} {'Min Value':<15} {'Epsilon':<12}")
print("-" * 70)
for fmt in fp_formats:
    print(f"{fmt['name']:<10} {fmt['bits']:<8} {fmt['max']:<15.2e} {fmt['min']:<15.2e} {fmt['eps']:<12.2e}")
print("=" * 70)

Use a tiny numerical example to see when reduced precision stops preserving small updates.

In [ ]:
# Demonstrate precision loss
print("\nPrecision Loss Demonstration:")
print("-" * 50)

# Small differences matter in gradients
a = torch.tensor(1.0, dtype=torch.float32)
small_diff = 1e-5

print(f"Base value: {a.item()}")
print(f"Adding {small_diff}:")

for dtype, name in [(torch.float32, 'FP32'), (torch.float16, 'FP16'), (torch.bfloat16, 'BF16')]:
    a_cast = a.to(dtype)
    result = (a_cast + small_diff).item()
    preserved = abs(result - (1.0 + small_diff)) < 1e-6
    status = "ok" if preserved else "X (precision lost)"
    print(f"  {name}: {result:.7f} {status}")

### Integer Quantization (INT8)

INT8 uses integers from -128 to 127 (signed) or 0 to 255 (unsigned).

To map floating point to integers, we use:
- **Scale** (s): Maps the float range to int range
- **Zero point** (z): Handles asymmetric distributions

**Quantization formula:**
$$q = \text{round}\left(\frac{x}{s}\right) + z$$

**Dequantization formula:**
$$x = s \cdot (q - z)$$

Where:
- $s = \frac{x_{max} - x_{min}}{q_{max} - q_{min}}$
- $z = q_{min} - \text{round}\left(\frac{x_{min}}{s}\right)$

In [ ]:
def quantize_tensor(x, num_bits=8, symmetric=True):
    """
    Quantize a tensor to fixed-point integers.
    
    Args:
        x: Input tensor (FP32)
        num_bits: Number of bits for quantization
        symmetric: If True, use symmetric quantization (zero point = 0)
        
    Returns:
        q: Quantized tensor (INT8)
        scale: Scale factor
        zero_point: Zero point offset
    """
    if symmetric:
        # Symmetric: [-max_abs, +max_abs] -> [-127, +127]
        q_max = 2**(num_bits - 1) - 1  # 127 for INT8
        max_abs = x.abs().max()
        scale = max_abs / q_max
        zero_point = 0
    else:
        # Asymmetric: [x_min, x_max] -> [0, 255]
        q_min, q_max = 0, 2**num_bits - 1  # 0, 255 for INT8
        x_min, x_max = x.min(), x.max()
        scale = (x_max - x_min) / (q_max - q_min)
        zero_point = q_min - torch.round(x_min / scale)
        zero_point = int(zero_point.clamp(q_min, q_max).item())
    
    # Quantize
    if symmetric:
        q = torch.round(x / scale).clamp(-q_max, q_max).to(torch.int8)
    else:
        q = torch.round(x / scale + zero_point).clamp(q_min, q_max).to(torch.uint8)
    
    return q, scale, zero_point


def dequantize_tensor(q, scale, zero_point):
    """Dequantize tensor back to floating point."""
    return scale * (q.float() - zero_point)


# Demonstrate quantization
print("Quantization Demonstration:")
print("=" * 60)

# Create a sample weight tensor (typical neural network weights)
weights_fp32 = torch.randn(100) * 0.5  # Weights often in [-1, 1]

print(f"\nOriginal FP32 weights:")
print(f"  Range: [{weights_fp32.min():.4f}, {weights_fp32.max():.4f}]")
print(f"  Memory: {weights_fp32.element_size() * weights_fp32.numel()} bytes")

# Symmetric quantization
q_sym, scale_sym, zp_sym = quantize_tensor(weights_fp32, symmetric=True)
weights_restored_sym = dequantize_tensor(q_sym, scale_sym, zp_sym)

print(f"\nSymmetric INT8 quantization:")
print(f"  Scale: {scale_sym:.6f}")
print(f"  Zero point: {zp_sym}")
print(f"  Memory: {q_sym.element_size() * q_sym.numel()} bytes (4x smaller!)")
print(f"  Quantization error (MSE): {F.mse_loss(weights_restored_sym, weights_fp32):.6f}")

# Asymmetric quantization
q_asym, scale_asym, zp_asym = quantize_tensor(weights_fp32, symmetric=False)
weights_restored_asym = dequantize_tensor(q_asym, scale_asym, zp_asym)

print(f"\nAsymmetric INT8 quantization:")
print(f"  Scale: {scale_asym:.6f}")
print(f"  Zero point: {zp_asym}")
print(f"  Quantization error (MSE): {F.mse_loss(weights_restored_asym, weights_fp32):.6f}")

Visualize quantization error for a simple range of values.

In [ ]:
# Visualize quantization effect
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Original distribution
axes[0].hist(weights_fp32.numpy(), bins=50, alpha=0.7, edgecolor='black', color='#3498db')
axes[0].set_title('Original FP32 Weights', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

# Quantized values
unique_q, counts = torch.unique(q_sym, return_counts=True)
axes[1].bar(unique_q.numpy(), counts.numpy(), alpha=0.7, edgecolor='black', color='#27ae60')
axes[1].set_title('Quantized INT8 Values', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Quantized Value')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

# Error distribution
error = (weights_restored_sym - weights_fp32).numpy()
axes[2].hist(error, bins=50, alpha=0.7, edgecolor='black', color='#e74c3c')
axes[2].set_title('Quantization Error', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Error')
axes[2].set_ylabel('Frequency')
axes[2].axvline(x=0, color='black', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Maximum absolute error: {abs(error).max():.6f}")
print(f"Mean absolute error: {abs(error).mean():.6f}")

## Part 3: Per-Channel vs Per-Tensor Quantization

Quantization granularity affects accuracy:

**Per-tensor**: One scale/zero_point for entire tensor
- Simple, efficient
- May lose accuracy if channels have different ranges

**Per-channel**: Different scale/zero_point for each output channel
- Better accuracy (each channel optimally quantized)
- Slightly more overhead
- Standard for weights in modern quantization

In [ ]:
def quantize_per_channel(weights, axis=0, num_bits=8):
    """
    Per-channel quantization for weight tensors.
    
    Args:
        weights: Weight tensor (out_channels, in_channels, ...)
        axis: Channel axis (typically 0 for output channels)
        num_bits: Quantization bits
    """
    q_max = 2**(num_bits - 1) - 1
    
    # Compute scale per channel
    dims_to_reduce = [i for i in range(weights.ndim) if i != axis]
    max_abs = weights.abs().amax(dim=dims_to_reduce, keepdim=True)
    scales = max_abs / q_max
    scales = scales.clamp(min=1e-8)  # Avoid division by zero
    
    # Quantize
    q = torch.round(weights / scales).clamp(-q_max, q_max).to(torch.int8)
    
    return q, scales.squeeze()


# Create a weight tensor with varying channel ranges
# Simulate real scenario: different channels have different magnitudes
torch.manual_seed(42)
out_channels, in_channels = 64, 128
weights_2d = torch.randn(out_channels, in_channels)

# Make some channels have larger values (common in real networks)
channel_scales = torch.rand(out_channels) * 2 + 0.5  # [0.5, 2.5]
weights_2d = weights_2d * channel_scales.unsqueeze(1)

print("Per-Channel vs Per-Tensor Quantization:")
print("=" * 60)

# Per-tensor quantization
q_tensor, scale_tensor, zp_tensor = quantize_tensor(weights_2d, symmetric=True)
restored_tensor = dequantize_tensor(q_tensor, scale_tensor, zp_tensor)
error_tensor = F.mse_loss(restored_tensor, weights_2d)

print(f"\nPer-Tensor: Single scale={scale_tensor:.4f}")
print(f"  MSE: {error_tensor:.6f}")

# Per-channel quantization
q_channel, scales_channel = quantize_per_channel(weights_2d, axis=0)
restored_channel = (q_channel.float() * scales_channel.unsqueeze(1))
error_channel = F.mse_loss(restored_channel, weights_2d)

print(f"\nPer-Channel: {out_channels} different scales")
print(f"  Scale range: [{scales_channel.min():.4f}, {scales_channel.max():.4f}]")
print(f"  MSE: {error_channel:.6f}")

improvement = (error_tensor - error_channel) / error_tensor * 100
print(f"\n-> Per-channel reduces error by {improvement:.1f}%!")

Inspect channel-wise quantization error.

In [ ]:
# Visualize channel-wise error
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Channel scales in original weights
original_ranges = weights_2d.abs().max(dim=1).values
axes[0].bar(range(out_channels), original_ranges.numpy(), alpha=0.7, color='#3498db')
axes[0].set_xlabel('Channel', fontsize=11)
axes[0].set_ylabel('Max Absolute Value', fontsize=11)
axes[0].set_title('Channel Value Ranges (Varying!)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Per-channel errors
errors_tensor = (restored_tensor - weights_2d).pow(2).mean(dim=1).sqrt()
errors_channel = (restored_channel - weights_2d).pow(2).mean(dim=1).sqrt()

x = np.arange(out_channels)
width = 0.4
axes[1].bar(x - width/2, errors_tensor.numpy(), width, label='Per-Tensor', alpha=0.7, color='#e74c3c')
axes[1].bar(x + width/2, errors_channel.numpy(), width, label='Per-Channel', alpha=0.7, color='#27ae60')
axes[1].set_xlabel('Channel', fontsize=11)
axes[1].set_ylabel('RMSE per Channel', fontsize=11)
axes[1].set_title('Quantization Error by Channel', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insight: Per-channel quantization adapts to each channel's range.")
print("This is why it's the standard for weight quantization!")

## Part 4: Building a Quantized Linear Layer

Let's implement a quantized linear layer from scratch to understand how quantized inference works.

In [ ]:
class QuantizedLinear(nn.Module):
    """
    Quantized linear layer with INT8 weights.
    
    Computation:
        y = (x_q - x_zp) * x_scale @ (w_q * w_scale).T + bias
        
    For simplicity, we keep activations in FP32 and only quantize weights.
    This is called "weight-only quantization".
    """
    
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Quantized weights (INT8) and scales (FP32)
        self.register_buffer('weight_q', torch.zeros(out_features, in_features, dtype=torch.int8))
        self.register_buffer('weight_scale', torch.zeros(out_features))
        
        if bias:
            self.register_buffer('bias', torch.zeros(out_features))
        else:
            self.bias = None
    
    @classmethod
    def from_float(cls, linear_fp32):
        """Create quantized layer from FP32 linear layer."""
        quant_layer = cls(
            linear_fp32.in_features,
            linear_fp32.out_features,
            bias=linear_fp32.bias is not None
        )
        
        # Quantize weights per-channel
        q, scales = quantize_per_channel(linear_fp32.weight.data, axis=0)
        quant_layer.weight_q.copy_(q)
        quant_layer.weight_scale.copy_(scales)
        
        if linear_fp32.bias is not None:
            quant_layer.bias.copy_(linear_fp32.bias.data)
        
        return quant_layer
    
    def forward(self, x):
        # Dequantize weights on-the-fly
        # In practice, this would be done more efficiently
        weight_fp32 = self.weight_q.float() * self.weight_scale.unsqueeze(1)
        
        output = F.linear(x, weight_fp32, self.bias)
        return output
    
    def memory_size(self):
        """Calculate memory footprint in bytes."""
        weight_bytes = self.weight_q.numel() * 1  # INT8 = 1 byte
        scale_bytes = self.weight_scale.numel() * 4  # FP32 = 4 bytes
        bias_bytes = self.bias.numel() * 4 if self.bias is not None else 0
        return weight_bytes + scale_bytes + bias_bytes


# Test the quantized layer
print("Quantized Linear Layer Test:")
print("=" * 60)

# Create FP32 layer
in_features, out_features = 512, 256
linear_fp32 = nn.Linear(in_features, out_features)

# Quantize
linear_int8 = QuantizedLinear.from_float(linear_fp32)

# Compare outputs
x = torch.randn(32, in_features)
y_fp32 = linear_fp32(x)
y_int8 = linear_int8(x)

print(f"\nFP32 layer:")
print(f"  Memory: {linear_fp32.weight.numel() * 4 + linear_fp32.bias.numel() * 4:,} bytes")
print(f"  Output range: [{y_fp32.min():.4f}, {y_fp32.max():.4f}]")

print(f"\nINT8 quantized layer:")
print(f"  Memory: {linear_int8.memory_size():,} bytes")
print(f"  Output range: [{y_int8.min():.4f}, {y_int8.max():.4f}]")

mse = F.mse_loss(y_int8, y_fp32)
rel_error = (y_int8 - y_fp32).abs().mean() / y_fp32.abs().mean() * 100
print(f"\nOutput difference:")
print(f"  MSE: {mse:.6f}")
print(f"  Relative error: {rel_error:.2f}%")

compression = (linear_fp32.weight.numel() * 4) / linear_int8.memory_size()
print(f"\nCompression ratio: {compression:.2f}x")

## Part 5: Post-Training Quantization (PTQ)

**Post-Training Quantization** converts a trained FP32 model to INT8 without retraining.

Two main approaches:
1. **Static quantization**: Pre-compute scales using calibration data
2. **Dynamic quantization**: Compute scales on-the-fly per batch

Let's implement both!

In [ ]:
# Define a simple CNN for CIFAR-10
class SimpleCNN(nn.Module):
    """Simple CNN for demonstration."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8 -> 4x4
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Load CIFAR-10
train_dataset, test_dataset = create_dataset('cifar10')
train_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=test_dataset,
    batch_size=128,
    num_workers=0,  # Use single-threaded loading to avoid multiprocessing warnings
    use_collate_fn=False  # CIFAR-10 doesn't need sequence padding
)

print("Model and dataset ready")

Define a shared training helper for the quantization experiments.

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=1e-3, device='cpu'):
    """Train a model."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        # Evaluate
        test_acc = evaluate_model(model, test_loader, device)
        print(f"Epoch {epoch+1}: Train Acc: {100*correct/total:.2f}%, Test Acc: {test_acc:.2f}%")
    
    return model


def evaluate_model(model, loader, device='cpu'):
    """Evaluate model accuracy."""
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return 100 * correct / total


# Train FP32 model
print("Training FP32 model...")
model_fp32 = SimpleCNN(num_classes=10)
model_fp32 = train_model(model_fp32, train_loader, test_loader, epochs=10, device=device)

fp32_acc = evaluate_model(model_fp32, test_loader, device)
print(f"\nFinal FP32 accuracy: {fp32_acc:.2f}%")

### Dynamic Quantization

**Dynamic quantization** quantizes weights ahead of time, but computes activation scales dynamically per batch.

Pros:
- No calibration data needed
- Simple to apply

Cons:
- Overhead from computing scales per batch
- Best for models dominated by linear layers (e.g., LSTMs)

In [ ]:
# PyTorch dynamic quantization
model_fp32_cpu = model_fp32.cpu()

model_dynamic = torch.quantization.quantize_dynamic(
    model_fp32_cpu,
    {nn.Linear},  # Quantize linear layers
    dtype=torch.qint8
)

print("Dynamic Quantization Results:")
print("=" * 60)

# Evaluate
dynamic_acc = evaluate_model(model_dynamic, test_loader, device='cpu')
print(f"FP32 accuracy:     {fp32_acc:.2f}%")
print(f"Dynamic INT8 acc:  {dynamic_acc:.2f}%")
print(f"Accuracy drop:     {fp32_acc - dynamic_acc:.2f}%")

### Static Quantization

**Static quantization** pre-computes both weight AND activation scales using calibration data.

Pros:
- Faster inference (no dynamic computation)
- Better for CNNs and models with convolutions

Cons:
- Requires calibration data (representative of inference data)
- More complex setup

In [ ]:
class QuantizableCNN(nn.Module):
    """CNN with quantization stubs for static quantization."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dequant = torch.quantization.DeQuantStub()
    
    def forward(self, x):
        x = self.quant(x)  # Quantize input
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.reshape(x.size(0), -1)  # Use reshape instead of view for quantized tensors
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)  # Dequantize output
        return x


def calibrate(model, loader, num_batches=100):
    """Run calibration to collect activation statistics."""
    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            model(images)


# Create quantizable model and copy weights
model_static = QuantizableCNN(num_classes=10)
model_static.load_state_dict(model_fp32_cpu.state_dict(), strict=False)

# Prepare for quantization
model_static.eval()
model_static.qconfig = torch.quantization.get_default_qconfig('fbgemm')
model_prepared = torch.quantization.prepare(model_static, inplace=False)

# Calibrate
print("Calibrating...")
calibrate(model_prepared, train_loader, num_batches=100)

# Convert to quantized model
model_quantized = torch.quantization.convert(model_prepared, inplace=False)

print("\nStatic Quantization Results:")
print("=" * 60)

# Evaluate
static_acc = evaluate_model(model_quantized, test_loader, device='cpu')
print(f"FP32 accuracy:     {fp32_acc:.2f}%")
print(f"Static INT8 acc:   {static_acc:.2f}%")
print(f"Accuracy drop:     {fp32_acc - static_acc:.2f}%")

## Part 6: FP16 / Mixed Precision

**FP16** (Half precision) offers:
- 2x memory reduction
- 2x faster on modern GPUs (Tensor Cores)
- Usually negligible accuracy loss

**Mixed Precision** training combines FP16 forward/backward with FP32 master weights:
- Forward pass: FP16 (fast)
- Backward pass: FP16 (fast)
- Weight update: FP32 (accurate)
- Uses loss scaling to prevent gradient underflow

In [ ]:
# Compare FP32 vs FP16 inference
if torch.cuda.is_available():
    # Create a fresh FP32 model for GPU (need separate copy since .half() modifies in-place)
    model_fp32_gpu = SimpleCNN(num_classes=10).to('cuda')
    model_fp32_gpu.load_state_dict(model_fp32_cpu.state_dict())
    
    # Create FP16 model from another copy
    model_fp16 = SimpleCNN(num_classes=10).half().to('cuda')
    model_fp16.load_state_dict({k: v.half() for k, v in model_fp32_cpu.state_dict().items()})
    
    # Evaluate FP16
    def evaluate_fp16(model, loader):
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in loader:
                images = images.half().cuda()  # Convert to FP16
                labels = labels.cuda()
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        return 100 * correct / total
    
    fp16_acc = evaluate_fp16(model_fp16, test_loader)
    
    print("FP32 vs FP16 Comparison:")
    print("=" * 60)
    print(f"FP32 accuracy: {fp32_acc:.2f}%")
    print(f"FP16 accuracy: {fp16_acc:.2f}%")
    print(f"Accuracy drop: {fp32_acc - fp16_acc:.2f}%")
    
    # Measure inference speed
    def measure_speed(model, dtype, num_batches=100):
        model.eval()
        x = torch.randn(128, 3, 32, 32, device='cuda', dtype=dtype)
        
        # Warmup
        for _ in range(10):
            _ = model(x)
        torch.cuda.synchronize()
        
        # Time
        start = time.time()
        for _ in range(num_batches):
            _ = model(x)
        torch.cuda.synchronize()
        elapsed = time.time() - start
        
        return elapsed / num_batches * 1000  # ms per batch
    
    time_fp32 = measure_speed(model_fp32_gpu, torch.float32)
    time_fp16 = measure_speed(model_fp16, torch.float16)
    
    print(f"\nInference Speed:")
    print(f"FP32: {time_fp32:.2f} ms/batch")
    print(f"FP16: {time_fp16:.2f} ms/batch")
    print(f"Speedup: {time_fp32/time_fp16:.2f}x")
else:
    print("CUDA not available. FP16 speedup is most significant on GPUs.")
    fp16_acc = fp32_acc  # Default for comparison

Run a mixed-precision training example with AMP.

In [ ]:
# Mixed Precision Training with AMP
if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler
    
    def train_mixed_precision(model, train_loader, epochs=5, lr=1e-3):
        """Train with automatic mixed precision."""
        model = model.cuda()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        scaler = GradScaler()
        
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            
            for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
                images, labels = images.cuda(), labels.cuda()
                
                optimizer.zero_grad()
                
                # Mixed precision forward pass
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                # Scaled backward pass
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                total_loss += loss.item()
            
            print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}")
        
        return model
    
    print("\nTraining with Automatic Mixed Precision (AMP):")
    print("=" * 60)
    
    # Train fresh model with AMP
    model_amp = SimpleCNN(num_classes=10)
    model_amp = train_mixed_precision(model_amp, train_loader, epochs=5)
    
    # Evaluate
    amp_acc = evaluate_model(model_amp, test_loader, device='cuda')
    print(f"\nAMP-trained model accuracy: {amp_acc:.2f}%")
    print("\nAMP provides 1.5-2x training speedup with minimal code changes!")

## Part 7: Quantization-Aware Training (QAT)

**Quantization-Aware Training** simulates quantization during training, allowing the model to adapt.

Key idea:
- Forward pass: Fake quantize (round then dequantize)
- Backward pass: Straight-through estimator (gradients pass through)

This is similar to 1-bit neural networks - the model learns to be robust to quantization noise!

In [ ]:
class FakeQuantize(torch.autograd.Function):
    """
    Fake quantization: quantize in forward, straight-through in backward.
    """
    @staticmethod
    def forward(ctx, x, scale, zero_point, q_min, q_max):
        # Quantize
        x_q = torch.round(x / scale + zero_point)
        x_q = x_q.clamp(q_min, q_max)
        # Dequantize
        x_dq = (x_q - zero_point) * scale
        return x_dq
    
    @staticmethod
    def backward(ctx, grad_output):
        # Straight-through estimator
        return grad_output, None, None, None, None


class QATLinear(nn.Module):
    """
    Linear layer with fake quantization for QAT.
    """
    def __init__(self, in_features, out_features, bias=True, num_bits=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.num_bits = num_bits
        self.q_min = -2**(num_bits - 1)
        self.q_max = 2**(num_bits - 1) - 1
        
        # Learnable scale (could also be fixed from calibration)
        self.register_buffer('weight_scale', torch.tensor(1.0))
    
    def forward(self, x):
        # Update scale based on weight range
        with torch.no_grad():
            max_abs = self.linear.weight.abs().max()
            self.weight_scale = max_abs / self.q_max
        
        # Fake quantize weights
        weight_q = FakeQuantize.apply(
            self.linear.weight,
            self.weight_scale,
            0,  # Symmetric quantization
            self.q_min,
            self.q_max
        )
        
        return F.linear(x, weight_q, self.linear.bias)


class QATCNN(nn.Module):
    """CNN with QAT-enabled layers."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = QATLinear(128 * 4 * 4, 256)  # QAT-enabled
        self.fc2 = QATLinear(256, num_classes)  # QAT-enabled
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# Train with QAT
print("Training with Quantization-Aware Training (QAT):")
print("=" * 60)

model_qat = QATCNN(num_classes=10)
model_qat = train_model(model_qat, train_loader, test_loader, epochs=10, device=device)

qat_acc = evaluate_model(model_qat, test_loader, device=device)
print(f"\nQAT model accuracy: {qat_acc:.2f}%")

## Part 8: Comparison and Analysis

In [ ]:
# Summary comparison
results = {
    'FP32 (baseline)': {'acc': fp32_acc, 'size': 4, 'method': 'None'},
    'Dynamic INT8': {'acc': dynamic_acc, 'size': 1, 'method': 'PTQ'},
    'Static INT8': {'acc': static_acc, 'size': 1, 'method': 'PTQ'},
    'QAT INT8': {'acc': qat_acc, 'size': 1, 'method': 'QAT'},
}

if torch.cuda.is_available():
    results['FP16'] = {'acc': fp16_acc, 'size': 2, 'method': 'Cast'}

print("\n" + "=" * 70)
print("QUANTIZATION COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Method':<20} {'Accuracy':<12} {'Size (bits)':<12} {'Acc Drop':<12} {'Type'}")
print("-" * 70)

for name, data in results.items():
    acc_drop = fp32_acc - data['acc']
    print(f"{name:<20} {data['acc']:>10.2f}% {data['size']:>10} {acc_drop:>10.2f}%   {data['method']}")

print("=" * 70)

Visualize how the mixed-precision results compare.

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = list(results.keys())
accuracies = [results[m]['acc'] for m in methods]
sizes = [results[m]['size'] for m in methods]

colors = ['#3498db', '#e74c3c', '#27ae60', '#f39c12', '#9b59b6'][:len(methods)]

# Accuracy comparison
bars = axes[0].bar(methods, accuracies, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim([min(accuracies) - 5, max(accuracies) + 2])
axes[0].axhline(y=fp32_acc, color='gray', linestyle='--', alpha=0.5, label='FP32 baseline')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)

# Size vs Accuracy scatter
for i, (m, acc, size) in enumerate(zip(methods, accuracies, sizes)):
    axes[1].scatter(size, acc, s=200, c=colors[i], edgecolor='black', linewidth=2, alpha=0.8)
    axes[1].annotate(m, (size, acc), xytext=(10, 5), textcoords='offset points', fontsize=9)

axes[1].set_xlabel('Bits per Weight', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Efficiency-Accuracy Trade-off', fontsize=14, fontweight='bold')
axes[1].set_xlim([0, 5])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 9: Practical Guidelines

### When to Use What

| Scenario | Recommended | Why |
|----------|-------------|-----|
| Quick deployment, good accuracy | **Static PTQ** | Best balance of effort and results |
| No calibration data available | **Dynamic PTQ** | Works without data |
| Accuracy-critical tasks | **QAT** | Model adapts to quantization |
| GPU inference | **FP16** | Simple, fast, minimal loss |
| Mobile/edge deployment | **INT8** | Best compression, dedicated hardware |
| LLM inference | **INT8 + FP16 hybrid** | Weights INT8, activations FP16 |

### Best Practices

1. **Always benchmark first**: Measure FP32 baseline before quantizing
2. **Use per-channel for weights**: Much better accuracy than per-tensor
3. **Calibrate on representative data**: Use data similar to inference
4. **Watch for sensitive layers**: First and last layers often need higher precision
5. **Profile on target hardware**: Speedup varies by platform

## Summary

### Key Takeaways

1. **Quantization** reduces model precision from FP32 to FP16/INT8, achieving 2-4x memory reduction and speedup.

2. **FP16** offers the best accuracy/speed tradeoff:
   - 2x smaller and faster
   - Usually <0.5% accuracy loss
   - Native GPU support (Tensor Cores)

3. **INT8** provides maximum compression:
   - 4x smaller than FP32
   - 1-2% typical accuracy loss
   - Dedicated hardware acceleration

4. **Post-Training Quantization (PTQ)** is simple:
   - Dynamic: No data needed, slower inference
   - Static: Needs calibration, faster inference

5. **Quantization-Aware Training (QAT)** minimizes accuracy loss:
   - Model learns to be robust to quantization
   - Best results for accuracy-critical tasks

6. **Per-channel quantization** is standard for weights:
   - Adapts to each channel's range
   - Much better than per-tensor

### Connections to Other Topics

- **1-Bit Neural Networks**: Extreme quantization using binary/ternary weights
- **Knowledge Distillation**: Compress by training smaller models
- **Pruning**: Remove weights instead of reducing precision
- **Speculative Decoding**: Accelerate inference with draft models

### Further Reading

- [PyTorch Quantization Documentation](https://pytorch.org/docs/stable/quantization.html)
- [A Survey of Quantization Methods for Efficient Neural Network Inference](https://arxiv.org/abs/2103.13630)
- [LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339)
- [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323)
- [AWQ: Activation-aware Weight Quantization](https://arxiv.org/abs/2306.00978)